# Statistics Villains: Web Scraping with pandas
## MAD 2502 — Applied Data

**Big idea:** Web scraping turns information embedded in a webpage into structured data that Python can analyze.

By the end, you should be able to inspect a webpage, check `robots.txt`, retrieve HTML, identify an HTML table, clean the resulting DataFrame, and decide whether scraping is an appropriate data source.

> **Lesson rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect


In [ ]:
import pandas as pd
import requests

PAGE_URL = (
    "https://people.clas.ufl.edu/mathguy/"
    "supervillain-statistics-playground/"
)


## 1. Understand

A browser downloads HTML and renders it into the page you see. Python can download the same HTML and search its structure.

For a table:

- `<table>` marks the table;
- `<tr>` marks a row;
- `<th>` marks a header cell;
- `<td>` marks a data cell.

Before scraping a site, prefer an API or downloadable file when one exists, and check the site's access rules.


In [ ]:
robots_url = "https://people.clas.ufl.edu/robots.txt"
robots_response = requests.get(robots_url, timeout=30)
print("robots.txt status:", robots_response.status_code)
print(robots_response.text[:800])


## 2. Predict

Before running the next cell, answer these questions:

1. Will `requests.get(...)` return the rendered webpage or the underlying response?
2. What type do you expect `response.text` to be?
3. If a page has three HTML tables, how many DataFrames do you think `pd.read_html(...)` returns?


In [ ]:
response = requests.get(PAGE_URL, timeout=30)
response.raise_for_status()
html = response.text

print("response type:", type(response))
print("HTML type:", type(html))
print("characters downloaded:", len(html))
print(html[:400])


## 3. Run

Let pandas locate the HTML tables. `pd.read_html(...)` returns a **list of DataFrames** because one page may contain several tables.


In [ ]:
tables = pd.read_html(html)

print("tables found:", len(tables))
for index, table in enumerate(tables):
    print(index, table.shape, list(table.columns))

villains = tables[0].copy()
display(villains.head())
print(villains.dtypes)


Compare this result with your prediction. Do not assume `tables[0]` is always correct on another website—inspect the available tables first.


## 4. Change

Web tables are written for people, not for Python. Change the table into a form that is easier to work with.

Before running the cleaning code, predict what happens to a heading such as `Insurance Cost`.


In [ ]:
villains.columns = [
    str(name).strip().lower().replace(" ", "_").replace("-", "_")
    for name in villains.columns
]

print(list(villains.columns))
display(villains.head())


In [ ]:
money_columns = [
    name for name in villains.columns
    if "insurance" in name or "cost" in name
]
print("possible money columns:", money_columns)

if money_columns:
    money = money_columns[0]
    cleaned = (
        villains[money].astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    villains[money] = pd.to_numeric(cleaned, errors="coerce")
    display(villains[[money]].head())


## 5. Use

Now use the cleaned table to answer actual questions rather than just demonstrating syntax.


In [ ]:
display(villains.describe(numeric_only=True))

if money_columns:
    money = money_columns[0]
    print("mean:", villains[money].mean())
    print("median:", villains[money].median())
    display(villains.sort_values(money, ascending=False).head(5))


In [ ]:
villains.to_csv("statistics_villains_clean.csv", index=False)
print("saved statistics_villains_clean.csv")


## 6. Practice

1. What does one row represent?
2. How many rows and columns were retrieved?
3. Choose one numeric column and compare its mean and median.
4. Find the five largest values in one numeric column.
5. Use `value_counts()` on a categorical column.
6. Explain why blindly using `tables[0]` could fail elsewhere.
7. Add one labeled graph that answers a question about the villains.


## 7. Connect

Web scraping adds a fragile step before ordinary data analysis: the webpage structure can change. For a capstone source, ask:

- Is there an API or CSV instead?
- Is automated access allowed?
- What does one row represent?
- Could the table structure change?
- Are you collecting only what you need?

**Connection:** After the table becomes a DataFrame, the workflow returns to the same pandas ideas from the previous tutorial.
